# SNOMED CT vocabulary — bulk download

SNOMED CT is the canonical clinical terminology — diagnoses,
procedures, findings, body sites, etc. `biodb.snomed` ships the
**OHDSI-flavoured `CONCEPT.csv`** (the concept dimension of the OMOP
Common Data Model's standardized vocabulary) as a GitHub Release
asset on this repo so downstream packages get a fast, versioned,
no-credentials download path.

This is a **bulk-only** source — there's no per-term REST API
wrapped here. For per-term semantics see [`biodb.ols`](04_ontology.ipynb)
(which covers SNOMED via OLS4) or [`biodb.umls`](https://uts.nlm.nih.gov/uts/)
(not yet wrapped — see the README's roadmap).

| Mode | Functions |
|---|---|
| **Bulk** | `download_concept_csv`, `load_concept_csv`, `get_concept_csv_path`, `is_available` |

In [1]:
from biodb import snomed

## 1. Where the asset lives

The release moved from `bschilder/synthlab` to `bschilder/bioDB` on
2026-05-18 — same bytes, identical SHA-256.

In [2]:
print(f"repo:    {snomed.GITHUB_REPO}")
print(f"release: {snomed.GITHUB_RELEASE_TAG}")
print(f"asset:   {snomed.GITHUB_ASSET_NAME}")
print(f"URL:     {snomed.SNOMED_RELEASE_URL}")

repo:    bschilder/bioDB
release: vocab-v1
asset:   CONCEPT.csv.gz
URL:     https://github.com/bschilder/bioDB/releases/download/vocab-v1/CONCEPT.csv.gz


## 2. Authentication strategies (private mirrors only)

The bioDB release is public, so the third strategy below is always
what runs. The other two are there for downstream consumers who
fork bioDB into a private mirror.

1. **`gh` CLI** — if installed and authenticated
2. **`GITHUB_TOKEN` / `GH_TOKEN`** environment variable
3. **Public URL** (default)

In [3]:
print(f"gh CLI authenticated: {snomed._gh_cli_available()}")
print(f"env token set:        {snomed._get_github_token() is not None}")

gh CLI authenticated: True
env token set:        False


## 3. Download + load

Smoke check — does the cached path exist already?

In [4]:
print(f"cached: {snomed.is_available()}")
print(f"cache dir: {snomed.CACHE_DIR}")

cached: False
cache dir: /Users/bschilder/.cache/biodb/snomed


Pull the asset (~29 MB compressed → ~175 MB CSV). Cached at
`~/.cache/biodb/snomed/CONCEPT.csv`; subsequent calls short-circuit.
Progress is reported via tqdm by default — pass `progress=False` to
silence.

```python
path = snomed.download_concept_csv()
```

`load_concept_csv` is the one-shot version — download (if needed)
plus `pd.read_csv` with the right dtype overrides.

```python
concept = snomed.load_concept_csv()
concept.shape  # → (~2.5 M rows, 10 columns)
concept[concept["vocabulary_id"] == "SNOMED"].head()
```

Both are skipped at notebook-execute time because the decompressed
CSV is 175 MB; running them locally is the right way to see them
work.

## 4. Schema

The columns map 1-to-1 to OHDSI's CDM concept table — see
<https://ohdsi.github.io/CommonDataModel/cdm54.html#CONCEPT>.

In [5]:
expected_columns = [
    "concept_id",  # integer primary key
    "concept_name",
    "domain_id",  # Condition / Drug / Procedure / Measurement / Device / Observation / …
    "vocabulary_id",  # SNOMED, RxNorm, LOINC, …  (filter to "SNOMED" for the SNOMED slice)
    "concept_class_id",
    "standard_concept",  # 'S' = standard, 'C' = classification, NaN = non-standard
    "concept_code",  # the original SNOMED ID (e.g. 38341003 for hypertensive disorder)
    "valid_start_date",
    "valid_end_date",
    "invalid_reason",
]
for col in expected_columns:
    print(col)

concept_id
concept_name
domain_id
vocabulary_id
concept_class_id
standard_concept
concept_code
valid_start_date
valid_end_date
invalid_reason
